# Ticketward: T4 smoke test on Colab (P2.21)

Runs the R-21 smoke test (`tw_ml.train.smoke`) on a Colab T4 **without** a published HF dataset.
Upload your local `records.jsonl` files instead.

**Before running:**

1. Runtime → Change runtime type → **T4 GPU**.
2. Edit the parameters cell: set `GIT_SHA` to the full 40-hex commit.
3. Have `data/generated/train/records.jsonl` and `data/generated/val/records.jsonl` ready to upload.

In [ ]:
# Parameters: edit GIT_SHA, then "Run all".
REPO_URL = "https://github.com/AkhilMittapalli/Ticketward"
GIT_SHA = "f6257ebe1daee13bdf9c8f83f09533a214bc4b48"
CONFIG = "ml/configs/sft_qwen3_1p7b.yaml"  # the provisional E3 winner
SMOKE_STEPS = 20
EXPECTED_TORCH = "2.11"
RUN_ROOT = "/content/tw-runs"

In [ ]:
# Clone at the pinned commit (removes any stale clone from a previous run).
import os, re, shutil, subprocess, sys

assert re.fullmatch(r"[0-9a-f]{40}", GIT_SHA), "set GIT_SHA to a 40-hex commit SHA"
os.chdir("/content")
if os.path.isdir("ticketward"):
    shutil.rmtree("ticketward")
subprocess.run(["git", "clone", "--quiet", REPO_URL, "ticketward"], check=True)
subprocess.run(["git", "-C", "ticketward", "checkout", "--quiet", "--detach", GIT_SHA], check=True)
head = subprocess.run(
    ["git", "-C", "ticketward", "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert head == GIT_SHA, f"checked out {head}, expected {GIT_SHA}"
os.chdir("ticketward")
print("repository at", head)

In [ ]:
# Platform check: Python 3.12/3.13, platform torch, T4.
import torch

print(sys.version)
assert sys.version_info[:2] in {(3, 12), (3, 13)}, "tw-ml needs Python 3.12 or 3.13"
torch_minor = ".".join(torch.__version__.split("+")[0].split(".")[:2])
assert torch_minor == EXPECTED_TORCH, f"torch {torch.__version__}, config expects {EXPECTED_TORCH}"
assert torch.cuda.is_available(), "select a GPU accelerator"
gpu = torch.cuda.get_device_name(0)
print(torch.__version__, torch.version.cuda, gpu)
assert "T4" in gpu, f"expected T4, got {gpu}"

In [ ]:
# Install pinned deps (no torch) then tw_ml.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
     "--require-hashes", "-r", "ml/requirements/train.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "./ml"],
    check=True,
)
print("tw_ml installed")

## Upload data

Run the next cell, then upload **two files** when prompted:

1. `train_records.jsonl` — your local `data/generated/train/records.jsonl`
2. `val_records.jsonl` — your local `data/generated/val/records.jsonl`

**Rename them before uploading** (or just upload them as-is and the cell handles both names).

In [ ]:
# Upload train and val records.jsonl files from your local machine.
from google.colab import files
import shutil
from pathlib import Path

train_dir = Path("data/generated/train")
val_dir = Path("data/generated/val")
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

print("Upload train/records.jsonl:")
uploaded = files.upload()
name = list(uploaded.keys())[0]
shutil.move(name, train_dir / "records.jsonl")
print(f"  → {train_dir / 'records.jsonl'} ({len(uploaded[name]):,} bytes)")

print("\nUpload val/records.jsonl:")
uploaded = files.upload()
name = list(uploaded.keys())[0]
shutil.move(name, val_dir / "records.jsonl")
print(f"  → {val_dir / 'records.jsonl'} ({len(uploaded[name]):,} bytes)")

In [ ]:
# Run the R-21 smoke test: 7 assertions on the T4.
os.makedirs(RUN_ROOT, exist_ok=True)
result = subprocess.run(
    [
        sys.executable, "-m", "tw_ml.train.smoke",
        "--config", CONFIG,
        "--steps", str(SMOKE_STEPS),
        "--run-root", RUN_ROOT,
        "--allow-unverified",
    ],
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
)
print(f"\nExit code: {result.returncode}")
assert result.returncode == 0, "smoke test failed — see output above"

In [ ]:
# Display the smoke report.
import json
from pathlib import Path

reports = sorted(Path(RUN_ROOT).rglob("smoke_report.json"))
for report_path in reports:
    print(f"\n{'='*60}")
    print(f"Report: {report_path}")
    print('='*60)
    report = json.loads(report_path.read_text())
    for check in report.get("checks", []):
        status = check.get("status", "?")
        marker = "✓" if status == "pass" else "✗" if status == "fail" else "~"
        print(f"  {marker} [{check.get('id', '?')}] {check.get('name', '?')}: {status}")
        if check.get("detail"):
            print(f"    {check['detail']}")
    print(f"\nPeak memory: {report.get('peak_memory_gib', '?')} GiB")
    print(f"Median s/step: {report.get('median_s_per_step', '?')}")
    print(f"Projected h/seed: {report.get('projected_hours_per_seed', '?')}")

if not reports:
    print("No smoke_report.json found — check the smoke test output above.")

In [ ]:
# Download the smoke report to your local machine.
for report_path in reports:
    files.download(str(report_path))